# CareFed-TrustLab walkthrough

This notebook uses synthetic data only. It is a research and teaching walkthrough, not a clinical system.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from carefed.data import SyntheticClinicalConfig, generate_synthetic_clinical_telemetry
from carefed.federated import run_federated_experiment
from carefed.experiments import export_experiment_bundle


## 1. Build a site-separated synthetic cohort
The dataset contains simulated hospital and home-care nodes, longitudinal vital-sign-style features, patient groups, and synthetic deterioration labels.

In [ ]:
frame = generate_synthetic_clinical_telemetry(SyntheticClinicalConfig(n_patients=360, n_sites=6, time_steps=12, seed=42))
frame.groupby('site_id').patient_id.nunique(), frame.deterioration_label.mean()


## 2. Run an auditable privacy-preserving federation
The model fits preprocessing on training patients only and selects the operating threshold on validation patients only.

In [ ]:
config = {'seed':42, 'rounds':4, 'local_epochs':2, 'learning_rate':0.01, 'privacy_enabled':True, 'clip_norm':1.0, 'noise_multiplier':0.35, 'aggregation':'secure_fedavg', 'attack':'none', 'output_dir':'../results/notebook_run'}
result = run_federated_experiment(frame, config)
result['metrics']


## 3. Inspect group audit and explanations
A gap is evidence to investigate, not a conclusion that the system is fair or unfair by itself.

In [ ]:
result['gaps'], result['explanations'].head(6)
export_experiment_bundle(result, '../results/notebook_run')
